# 00 — Set up the GMS stores

The chapter notebooks load **trained GMS stores** from `code/data/`
(`gms_banking_store`, `gms_policy_store`, `gms_regulatory_store`, and the
GEODE policy-RAG store). Those are build artifacts: **not committed and not
shipped in the package** (git-ignored, like all trained weights). A fresh
checkout has none of them, so most chapters fail with *"failed to load GMS
... store"* until this notebook has been run.

**What you need**

- The licensed **`knowlytix`** substrate: `pip install knowlytix` plus a
  developer license from https://knowlytix.ai/signup/ (key at
  `~/.knowlytix/license.key`). Without it no store can be built.
- `torch`.

**Two tiers**

1. **Tier 1 — CPU, no LLM.** The banking substrate store, its calibrated
   gate thresholds, the policy entity index and the regulatory guard. This
   is what most chapters (and the capstone tests) need. A few minutes.
2. **Tier 2 — GPU + Qwen.** The GEODE self-corrected policy-RAG store, used
   by the retrieval/answering chapters. Downloads Qwen2.5-3B-Instruct and
   expects a CUDA GPU.

Every stage is **idempotent** — skipped when its output already exists — so
you can re-run this notebook freely.

## 1. Bootstrap: locate `knowlytix` and this topic's code/ dir

In [ ]:
import importlib.util, os, sys

# Locate this topic's code/ dir, robust to the working directory.
_cwd = os.getcwd()
for REPO in (_cwd, os.path.join(os.path.dirname(_cwd), "code"), os.path.join(_cwd, "code")):
    if os.path.isdir(os.path.join(REPO, "agentlab")):
        break
else:
    REPO = os.environ.get("AGENTLAB_REPO", _cwd)
for p in (REPO, os.path.join(REPO, 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)

if importlib.util.find_spec('knowlytix') is None:
    raise ModuleNotFoundError(
        'knowlytix not found. Install it (`pip install knowlytix`, licensed) '
        'and put your key at ~/.knowlytix/license.key — see '
        'https://knowlytix.ai/signup/.')

import torch
print('torch:', torch.__version__, '| device:',
      'cuda' if torch.cuda.is_available() else 'cpu')
print('repo :', REPO)

## 2. Idempotent stage runner

In [ ]:
import subprocess, sys, os

def stage(title, script, args=(), produces=()):
    """Run scripts/<script> unless every path in `produces` already exists.
    Streams the script's output; raises on non-zero exit."""
    produces = list(produces)
    if produces and all(os.path.exists(os.path.join(REPO, p)) for p in produces):
        print(f'\u2713 {title}: already built \u2014 skipping')
        return
    cmd = [sys.executable, os.path.join(REPO, 'scripts', script), *map(str, args)]
    print(f'\u25b6 {title}: python scripts/{script} ' + ' '.join(map(str, args)))
    r = subprocess.run(cmd, cwd=REPO, env=os.environ.copy())
    if r.returncode:
        raise RuntimeError(f'{script} failed (exit {r.returncode})')
    print(f'\u2713 {title}: done')

## 3. Tier 1 — CPU stores (required)

No LLM and no GPU. The banking store is the Chapter-16 substrate (ENM,
tension, plausibility gate); calibrating it writes the gate thresholds the
capstone reads. The policy and regulatory stores back `search_policy` and
`flag_regulatory`.

In [ ]:
stage('Banking substrate store', 'retrain_gms_banking.py',
      produces=['data/gms_banking_store/model.pt'])

stage('Calibrate plausibility/contradiction thresholds', 'calibrate_gms_thresholds.py',
      produces=['data/gms_banking_store/calibration.json'])

stage('Policy entity index (Graph-RAG)', 'build_policy_rag_store.py',
      produces=['data/gms_policy_store/model.pt'])

stage('Regulatory guard store', 'build_regulatory_guard_store.py',
      produces=['data/gms_regulatory_store/model.pt'])

## 4. Verify the Tier-1 stores load

In [ ]:
import json, os

for name in ('gms_banking_store', 'gms_policy_store', 'gms_regulatory_store'):
    p = os.path.join(REPO, 'data', name)
    print(f"{name:24} {'OK ' if os.path.isdir(p) else 'MISSING'}")

cal = os.path.join(REPO, 'data', 'gms_banking_store', 'calibration.json')
if os.path.isfile(cal):
    theta = json.load(open(cal))['plausibility_gate']['threshold']
    print('calibrated plausibility threshold:', theta)

## 5. Tier 2 — GEODE policy-RAG store (optional; GPU + Qwen)

The retrieval/answering chapters bind queries through a GEODE
self-corrected policy graph. Building it runs Qwen2.5-3B-Instruct and
**expects a CUDA GPU**; it also downloads the model (~6 GB). Set
`RUN_TIER2 = True` to build it.

In [ ]:
RUN_TIER2 = False  # set True to build the GEODE policy-RAG store (GPU + Qwen)

if RUN_TIER2:
    stage('GEODE policy-RAG store (loads Qwen)', 'build_geode_rag_store.py',
          produces=['data/gms_policy_store_geode/model.pt'])
    print('\nTier 2 complete.')
else:
    print('RUN_TIER2 is False \u2014 Tier-1 stores only. The retrieval/answering '
          'chapters and the end-to-end capstone tests need Tier 2.')

## Done

The stores live under `code/data/` (git-ignored, never packaged). Open the
chapter notebooks — they load these directly. Re-run this notebook any time;
existing stores are skipped.